In [41]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

start_date = datetime.now() - timedelta(days = 5*365)
df = yf.download("BTC-USD", start=start_date, end=datetime.now(), interval = "1d", auto_adjust=True)

if not df.empty:
    df.to_csv("BTC-USD_5_years")
    df.head()
else:
    print("No data found")

[*********************100%***********************]  1 of 1 completed


In [42]:
df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
df['Daily Return'] = df['Close'].pct_change() #Daily Return
df['MA_7'] = df['Close'].rolling(window=7).mean() #MA 7 days period
df['MA_30'] = df['Close'].rolling(window=30).mean() #MA 30 days period
df['Volatility_30'] = df['Daily Return'].rolling(window=30).std() #Moving volatility 30d period

#

In [43]:

def compute_rsi(close: 'pd.Series', length: int = 14) -> 'pd.Series':
    """
    Compute the Relative Strength Index (RSI) using Wilder's smoothing.
    Equivalent to pandas_ta.rsi(length=length).
    """
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # Wilder's smoothing == EMA with alpha = 1/length
    avg_gain = gain.ewm(alpha=1 / length, adjust=False, min_periods=length).mean()
    avg_loss = loss.ewm(alpha=1 / length, adjust=False, min_periods=length).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def compute_macd(close: 'pd.Series', fast: int = 12, slow: int = 26, signal: int = 9) -> 'pd.DataFrame':
    """
    Compute MACD (Moving Average Convergence Divergence).
    Returns a DataFrame with columns: MACD, Signal, Histogram
    Equivalent to pandas_ta.macd(fast=fast, slow=slow, signal=signal).
    """
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()

    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line

    # Return as DataFrame matching pandas_ta output column names
    result = pd.DataFrame({
        'MACD_12_26_9': macd_line,
        'MACDh_12_26_9': histogram,
        'MACDs_12_26_9': signal_line,
    })
    return result

df['RSI'] = compute_rsi(df['Close'])
macd_df = compute_macd(df['Close'], fast=12, slow=26, signal=9)
df = df.join(macd_df)

In [44]:
df['target_up_tomorrow'] = (df['Daily Return'].shift(-1) > 0)
df.dropna()
df_cleaned = df.dropna().copy()

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

features = ['Daily Return', 'MA_7', 'MA_30', 'Volatility_30', 'RSI', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9']

# Creating pipeline and column transformer
preprocessor = ColumnTransformer(
    transformers=[('scale_indicators', StandardScaler(), features)],
    remainder='drop'
)

gb_pipeline = Pipeline(steps =[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))
])

In [46]:
X = df_cleaned[features]

y = df_cleaned['target_up_tomorrow'].astype(int)

# financial time series can't use train test split because of data leakage caused by random state shuffle
# A chronological split (use the first 80% for training, last 20% for testing)
split_index = int(len(X) * 0.80)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [47]:
from sklearn.metrics import classification_report, accuracy_score

gb_pipeline.fit(X_train, y_train)

y_pred = gb_pipeline.predict(X_test)

print(f"Out-of-Sample Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

Out-of-Sample Accuracy: 0.4819

              precision    recall  f1-score   support

           0       0.52      0.29      0.37       190
           1       0.47      0.70      0.56       169

    accuracy                           0.48       359
   macro avg       0.49      0.49      0.47       359
weighted avg       0.49      0.48      0.46       359



In [48]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# Initialize the splitter
tscv = TimeSeriesSplit(n_splits=5)

# Use cross_val_score to see how your pipeline performs across 5 different market periods
scores = cross_val_score(gb_pipeline, X, y, cv=tscv, scoring='accuracy')

print(f"Accuracy for each market phase: {scores}")
print(f"Average Accuracy: {scores.mean():.4f}")

Accuracy for each market phase: [0.52508361 0.5083612  0.52508361 0.48829431 0.5083612 ]
Average Accuracy: 0.5110
